# Task 4

> *Run the above various algorithms in 3) on MSH-WSD dataset and report the average accuracy on all acronyms
> and average accuracy on all regular terms.*

In [46]:
import nltk
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

from pathlib import Path
import html
import re
import random
import json
import time

from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
from nltk.wsd import lesk

from pywsd._wordnet import wn as pwn
from pywsd.lesk import original_lesk, simple_lesk, adapted_lesk, cosine_lesk
from pywsd.similarity import max_similarity
from pywsd.baseline import max_lemma_count
from pywsd.utils import lemmatize
import wn.compat.sensekey as sensekey  

STOPWORDS = set(stopwords.words("english"))

Let's open the file `benchmark_mesh.txt` to extract all the ambiguous words of the MSH-WSD dataset with
their possible meanings (CUIs), and store them in a dictionary as in Task 2.

In [47]:
DATA_DIR = Path(r"C:\Users\flona\Documents\nlp_project\nlp_project_26\data\MSHCorpus")

concept_map = {}

with open(DATA_DIR / "benchmark_mesh.txt", encoding="iso-8859-1") as f:
    for line in f:
        parts = line.strip().split("\t")   
        word = parts[0]                   
        cuis = parts[1:]               
        concept_map[word] = {f"M{i}": cui for i, cui in enumerate(cuis, start=1)}

print("Number of ambiguous words:", len(concept_map))
print("AA ->", concept_map["AA"])
print("Ca ->", concept_map["Ca"])

Number of ambiguous words: 203
AA -> {'M1': 'C0002520', 'M2': 'C0001972'}
Ca -> {'M1': 'C0006675', 'M2': 'C0006754', 'M3': 'C0019564', 'M4': 'C0006823'}


Now we read, for each word, its file with all the abstracts where the word appears, together with the correct
meaning of the word in that context. As in Task 2, but we also keep the text **with** the `<e>...</e>` tags
(`tagged`), because later we need to know where the target word is in order to take the words around it.

In [48]:
def load_instances(word):
    """Read all the sentences (instances) of one ambiguous word."""

    path = DATA_DIR / f"{word}_pmids_tagged.arff"
    lines = path.read_text(encoding="iso-8859-1").splitlines()

    # The sentences start right after the line "@DATA"
    start = next(i for i, l in enumerate(lines) if l.startswith("@DATA")) + 1

    instances = []
    for line in lines[start:]:
        line = line.strip()
        if not line:                       
            continue

        pmid, rest = line.split(",", 1)    # text before the FIRST comma = PMID
        text, label = rest.rsplit(",", 1)  # text after the LAST comma = label (M1, M2...)
        text = html.unescape(text.strip('"'))   # remove quotes, convert &lt; -> <

        # Remember which word was marked, then remove the <e> </e> tags
        marked = re.search(r"<e>(.+?)</e>", text)
        target = marked.group(1) if marked else word
        context = re.sub(r"</?e>", "", text)

        instances.append({
            "word": word,
            "pmid": pmid,
            "target": target,                          # e.g. "cements"
            "tagged": text,                            # e.g. "The <e>cements</e> were applied."
            "context": context,                        # clean sentence
            "label": label.strip(),                    # e.g. "M1"
            "cui": concept_map[word][label.strip()],   # correct sense, e.g. "C0002520"
        })
    return instances

We evaluate the algorithms on the words of Task 2 (both selections together, 39 words), because they are the
only ones with a **manual CUI → WordNet mapping**, i.e. the correct answers translated into WordNet senses.
First we split all the words of the dataset into acronyms and regular terms with the same rule as Task 2
(the task asks for a separate accuracy for each group); then we keep the ones that have a mapping.

In [49]:
# Rule: a word is an acronym if it has 2+ capital letters or contains a digit
# (e.g. "AA", "BPD", "CCl4"). Multi-word names ("B-Cell Leukemia") are never acronyms.
def is_acronym(word):
    if " " in word:
        return False
    n_upper = sum(c.isupper() for c in word)
    has_digit = any(c.isdigit() for c in word)
    return n_upper >= 2 or has_digit

# Manual corrections (checked by hand in Task 2)
MANUAL_ACRONYMS = {"Ca", "Ala", "Epi", "Fe", "dC", "pI", "Staph", "Strep"}
MANUAL_TERMS    = {"ERUPTION", "LABOR", "NEUROFIBROMATOSIS", "THYMUS", "Gamma-Interferon"}
EXCLUDED        = {"EGG", "STEM", "Orf"}   # can be both a term and an acronym

all_words = sorted(w for w in concept_map if w not in EXCLUDED)
all_acronyms = [w for w in all_words if (is_acronym(w) or w in MANUAL_ACRONYMS) and w not in MANUAL_TERMS]
all_terms    = [w for w in all_words if w not in all_acronyms]

print("Acronyms:", len(all_acronyms))
print("Terms   :", len(all_terms))

Acronyms: 108
Terms   : 92


In [52]:
# mapping of MSH-WSD senses (CUIs) to WordNet synsets, created by hand
MAPPING = {
    # ---------------- acronyms ----------------
    "ADP":   {"C0001459": ["adenosine_diphosphate.n.01"],      # Adenosine Diphosphate
              "C0004374": ["automatic_data_processing.n.01"]}, # Automatic Data Processing
    "CAD":   {"C0011905": [],                                  # Computer Assisted Diagnosis (WN has only "design")
              "C1956346": []},                                 # Coronary Artery Disease
    "CAM":   {"C0007578": [],                                  # Cell Adhesion Molecules
              "C0178551": []},                                 # chorioallantoic membrane
    "CDA":   {"C0092801": [],                                  # cladribine
              "C0002876": []},                                 # Congenital dyserythropoietic anemia
    "Ca":    {"C0006675": ["calcium.n.01"],                    # calcium
              "C0006754": ["california.n.01"],                 # California
              "C0019564": [],                                  # Hippocampus
              "C0006823": []},                                 # Canada
    "DDD":   {"C0026256": [],                                  # mitotane
              "C0011037": []},                                 # Dichlorodiphenyldichloroethane
    "DON":   {"C0028652": [],                                  # Nurse Administrator
              "C0012020": []},                                 # Diazooxonorleucine
    "SARS":  {"C1175175": ["severe_acute_respiratory_syndrome.n.01"],  # the disease
              "C1175743": []},                                 # the coronavirus
    "Staph": {"C0038160": [],                                  # Staphylococcal Infections
              "C0038170": ["staphylococcus.n.01"]},            # Genus staphylococcus
    "TNT":   {"C0077404": [],                                  # Troponin T
              "C0041070": ["tnt.n.01"]},                       # Trinitrotoluene
    "ALS": {"C0003372": [],                                         # Antilymphocyte Serum
            "C0002736": ["amyotrophic_lateral_sclerosis.n.01"]},    # Amyotrophic Lateral Sclerosis
    "ANA": {"C0002463": [],                                         # American Nurses' Association
            "C0003243": []},                                        # Antinuclear Antibodies
    "CI":  {"C0022326": [],                                         # Ivory Coast
            "C0008107": []},                                        # Chile
    "CLS": {"C0265252": [],                                         # Coffin-Lowry syndrome
            "C0343084": []},                                        # Capillary Leak Syndrome
    "FAS": {"C0015683": [],                                         # Fatty-acid synthase
            "C0015923": ["fetal_alcohol_syndrome.n.01"]},           # Fetal Alcohol Syndrome
    "GAG": {"C0017346": [],                                         # GAG Gene
            "C0017973": []},                                        # Glycosaminoglycans
    "MRS": {"C0025235": [],                                         # Melkersson-Rosenthal Syndrome
            "C0024487": []},                                        # Magnetic Resonance Spectroscopy
    "PVC": {"C0151636": ["premature_ventricular_contraction.n.01"], # Premature ventricular contractions
            "C0032624": ["polyvinyl_chloride.n.01"]},               # Polyvinyl Chloride
    "RA":  {"C0002893": [],                                         # Refractory anemias
            "C0034625": ["radium.n.01"],                            # radium
            "C0003873": []},                                        # Rheumatoid Arthritis
    "US":  {"C0041703": ["united_states.n.01"],                     # United States
            "C0041618": []},                                        # Ultrasonography
    # ---------------- regular terms ----------------
    "B-Cell Leukemia": {"C2004493": [],                        # Leukemia, B-Cell
                        "C0023434": []},                       # Chronic Lymphocytic Leukemia
    "Borrelia":    {"C0024198": [],                            # Lyme Disease
                    "C0006033": ["borrelia.n.01"]},            # Borrelia (bacterium)
    "Cilia":       {"C0008778": ["cilium.n.01"],               # Cilia (cell projection)
                    "C0015422": ["eyelash.n.01"]},             # Eyelash
    "Follicles":   {"C0221971": [],                            # Hair follicle
                    "C0018120": ["follicle.n.01"]},            # Ovarian Follicle (closest to WN gloss)
    "Ganglion":    {"C0017067": ["ganglion.n.01"],             # Ganglia (neural)
                    "C1258666": []},                           # Myxoid cyst
    "Nurse":       {"C0028661": ["nurse.n.01", "nurse.v.03"],  # Nurses (the profession)
                    "C0006147": ["breastfeed.v.01"]},          # Breast Feeding
    "Pneumocystis": {"C0032305": [],                           # (retired CUI, name not available)
                     "C0597258": []},                          # Genus Pneumocystis
    "Respiration": {"C0035203": ["respiration.n.02", "breathing.n.01"],  # breathing
                    "C0282636": ["respiration.n.01"]},         # Cell Respiration (metabolic)
    "Sodium":      {"C0037473": ["sodium.n.01"],               # sodium (element)
                    "C0037570": []},                           # Sodium, Dietary
    "Synapsis":    {"C0039062": [],                            # Synapses (neuron junction)
                    "C0598501": ["synapsis.n.01"]},            # Chromosome Pairing
    "Callus":     {"C0006767": ["callus.n.02"],                     # Bone callus
                   "C0376154": ["callosity.n.01"]},                 # Skin callus
    "Cell":       {"C0007634": ["cell.n.02"],                       # Cells (biology)
                   "C1136359": ["cellular_telephone.n.01"]},        # Cellular Phone
    "Cement":     {"C1706094": ["cement.n.04", "cement.n.03"],      # Dental Cement Material
                   "C0011343": ["cementum.n.01"]},                  # Dental Cementum
    "ERUPTION":   {"C0015230": ["eruption.n.02"],                   # Exanthema (skin rash)
                   "C1533692": ["eruption.n.06"]},                  # Tooth eruption
    "Exercises":  {"C0452240": ["exercise.n.03"],                   # Physical therapy exercises
                   "C0015259": ["exercise.n.01"]},                  # Exercise
    "Medullary":  {"C0001629": ["medullary.a.03"],                  # Adrenal Medulla
                   "C0025148": ["medullary.a.02"]},                 # Medulla Oblongata
    "Milk":       {"C0026131": ["milk.n.01"],                       # Milk (body substance)
                   "C0026140": ["milk.n.02"]},                      # Milk, Human
    "Phosphorus": {"C0080014": [],                                  # Dietary Phosphorus
                   "C0031705": ["phosphorus.n.01"]},                # phosphorus (element)
    "Ventricles": {"C0018827": ["ventricle.n.02"],                  # Heart Ventricle
                   "C0007799": ["ventricle.n.01"]},                 # Cerebral Ventricles
}

ACRONYMS = [w for w in all_acronyms if w in MAPPING]
TERMS    = [w for w in all_terms if w in MAPPING]

print(len(ACRONYMS), "acronyms:", ACRONYMS)
print(len(TERMS), "terms:", TERMS)

20 acronyms: ['ADP', 'ALS', 'ANA', 'CAD', 'CAM', 'CDA', 'CI', 'CLS', 'Ca', 'DDD', 'DON', 'FAS', 'GAG', 'MRS', 'PVC', 'RA', 'SARS', 'Staph', 'TNT', 'US']
19 terms: ['B-Cell Leukemia', 'Borrelia', 'Callus', 'Cell', 'Cement', 'Cilia', 'ERUPTION', 'Exercises', 'Follicles', 'Ganglion', 'Medullary', 'Milk', 'Nurse', 'Phosphorus', 'Pneumocystis', 'Respiration', 'Sodium', 'Synapsis', 'Ventricles']


Another mapping is needed, between the WordNet version used by NLTK and the one used by pywsd. 
Our `MAPPING` gives the correct answers with the NLTK names, while the pywsd algorithms answer with their own IDs, so we need to translate them. To do that we use the **sense key**, a code that identifies the same sense in both versions.

Before looking a word up, pywsd lemmatizes it and distinguishes upper and lower case, so the same word can give different senses. The function "pywsd_form" tries a few spellings and keeps the one that finds the most senses of our `MAPPING`.

In [ ]:
def wordnet_senses(word):
    """NLTK WordNet senses of an MSH-WSD word, used only by NLTK Lesk"""
    w = word.lower().replace(" ", "_")
    senses = wn.synsets(w)
    if not senses:
        base = wn.morphy(w)
        if base:
            senses = wn.synsets(base)
    return senses

get_key = sensekey.sense_key_getter("oewn:2024")   # function: pywsd sense -> sense key

def to_nltk(pywsd_synset):
    """Translate a pywsd synset (e.g. oewn-01460746-n) into its NLTK name (e.g. cilium.n.01), through the sense key."""
    if pywsd_synset is None:
        return None
    for sense in pywsd_synset.senses():
        key = get_key(sense)
        if key:
            try:
                return wn.lemma_from_key(key).synset().name()
            except Exception:
                continue
    return None

def pywsd_form(word):                                                  
    """Spelling of the word that lets pywsd see most of the senses used in MAPPING."""
    mapped = {s for synsets in MAPPING[word].values() for s in synsets}  
    best_form, best_score = word, (-1, -1)
    for form in dict.fromkeys([word, word.lower(), word.capitalize(), word.upper()]):
        found = {to_nltk(s) for s in pwn.synsets(lemmatize(form))}
        score = (len(found & mapped), len(found))
        if score > best_score:
            best_form, best_score = form, score
    return best_form


for word in ACRONYMS + TERMS:
    print(f"{word:16} -> {pywsd_form(word)!r}")

ADP              -> 'ADP'
ALS              -> 'ALS'
ANA              -> 'ANA'
CAD              -> 'cad'
CAM              -> 'CAM'
CDA              -> 'CDA'
CI               -> 'CI'
CLS              -> 'CLS'
Ca               -> 'ca'
DDD              -> 'DDD'
DON              -> 'DON'
FAS              -> 'FAS'
GAG              -> 'GAG'
MRS              -> 'MRS'
PVC              -> 'PVC'
RA               -> 'ra'
SARS             -> 'SARS'
Staph            -> 'Staph'
TNT              -> 'TNT'
US               -> 'US'
B-Cell Leukemia  -> 'B-Cell Leukemia'
Borrelia         -> 'Borrelia'
Callus           -> 'Callus'
Cell             -> 'Cell'
Cement           -> 'Cement'
Cilia            -> 'Cilia'
ERUPTION         -> 'ERUPTION'
Exercises        -> 'exercises'
Follicles        -> 'follicles'
Ganglion         -> 'Ganglion'
Medullary        -> 'Medullary'
Milk             -> 'milk'
Nurse            -> 'Nurse'
Phosphorus       -> 'phosphorus'
Pneumocystis     -> 'Pneumocystis'
Respiration      -

The similarity-based methods compare every context word with every sense of the target word, so on a whole abstract they are very slow. We therefore use a **context window**: only the 10 words before and the 10 words after
the target word, found thanks to the `<e>...</e>` tags. The closest words are usually also the most useful
clues. The same window is used for all the methods, so the comparison is fair.

In [ ]:
WINDOW = 10    

def context_window(tagged_text, n=WINDOW):
    """Return the n words before the target, the target itself and the n words after it."""
    if "<e>" not in tagged_text:                
        return re.sub(r"</?e>", "", tagged_text)

    before, rest = tagged_text.split("<e>", 1)      
    target, after = rest.split("</e>", 1)             

    words_before = re.findall(r"\w+", before)[-n:]    
    words_after = re.findall(r"\w+", re.sub(r"</?e>", "", after))[:n]   

    return " ".join(words_before + [target] + words_after)            


# Check on one abstract
example = load_instances("AA")[0]
print("Full abstract :", len(example["context"].split()), "words")
print("Window        :", context_window(example["tagged"]))

Full abstract : 262 words
Window        : effect of the duration of prefeeding on prececal amino acid AA digestibility of wheat distillers dried grains with solubles DDGS in


Now we put all the methods in a single dictionary `METHODS`, so that we can run them with one loop, and we
write a function `predict` that runs one method on one context window and returns its answer as an NLTK
synset name.

The methods are the ones of Task 3 (Lesk family, similarity-based methods and Max lemma count), plus:
- **NLTK Lesk**, the algorithm of Task 2, as a reference. It uses the NLTK WordNet directly, so its answer
  does not need to be translated;
- **Resnik (fixed)**: in Task 3 we found that pywsd sorts the Resnik ranking in ascending order and returns
  the **least** similar sense. `resnik_fixed` takes the full ranking and returns the sense with the
  **highest** score. We keep both versions to see how much the bug changes the results.

In [ ]:
def resnik_fixed(context, word):
    """Resnik with the ranking sorted correctly (highest score = most similar)."""
    ranking = max_similarity(context, word, option="res", best=False)
    if not ranking:
        return None
    return max(ranking, key=lambda x: x[0])[1]


METHODS = {
    "Original Lesk":    lambda ctx, w: original_lesk(ctx, w),
    "Simple Lesk":      lambda ctx, w: simple_lesk(ctx, w),
    "Adapted Lesk":     lambda ctx, w: adapted_lesk(ctx, w),
    "Cosine Lesk":      lambda ctx, w: cosine_lesk(ctx, w),
    "Path":             lambda ctx, w: max_similarity(ctx, w, option="path"),
    "Wu-Palmer":        lambda ctx, w: max_similarity(ctx, w, option="wup"),
    "Leacock-Chodorow": lambda ctx, w: max_similarity(ctx, w, option="lch"),
    "Resnik (pywsd)":   lambda ctx, w: max_similarity(ctx, w, option="res"),
    "Resnik (fixed)":   resnik_fixed,
    "Jiang-Conrath":    lambda ctx, w: max_similarity(ctx, w, option="jcn"),
    "Lin":              lambda ctx, w: max_similarity(ctx, w, option="lin"),
    "Max lemma count":  lambda ctx, w: max_lemma_count(w),
}
ALL_METHODS = ["NLTK Lesk"] + list(METHODS)


def predict(method, context, word):                                   
    """Run one method on one context and return its answer as an NLTK synset name (or None)."""
    try:
        if method == "NLTK Lesk":
            senses = wordnet_senses(word)
            if not senses:
                return None
            answer = lesk(context.lower().split(), word.lower(), synsets=senses)
            return answer.name() if answer else None
        answer = METHODS[method](context, pywsd_form(word))           
        return to_nltk(answer)
    except Exception:
        return None

For each word we take 10 random samples.
Sampling the same number of abstracts per sense keeps the senses balanced: if we sampled at random, a word
with 150 abstracts of one sense and 50 of the other could end up with almost only the first sense, and a
method that always chooses it would look better than it is. The sample is fixed (`seed=0`), so it is the same
every time the notebook is run. In total we use 810 abstracts.

In [56]:
N_PER_SENSE = 10   # abstracts sampled for each sense of each word

def sample_instances(word, n=N_PER_SENSE, seed=0):
    """Random sample of n abstracts for each sense (CUI) of the word."""
    rng = random.Random(seed)
    by_sense = {}
    for inst in load_instances(word):
        by_sense.setdefault(inst["cui"], []).append(inst)
    sample = []
    for cui in sorted(by_sense):
        sample += rng.sample(by_sense[cui], min(n, len(by_sense[cui])))
    return sample


n_abstracts = sum(len(sample_instances(w)) for w in ACRONYMS + TERMS)
print("Abstracts in the sample:", n_abstracts)

Abstracts in the sample: 810


Now we run all the methods on every abstract of the sample. For each abstract we store its correct sense (CUI)
and the answer of every method; the accuracy is computed later from these stored answers, without running the
algorithms again.

In [58]:
CACHE = Path("task4_predictions.json")   

predictions = json.loads(CACHE.read_text()) if CACHE.exists() else {}

for word in ACRONYMS + TERMS:
    # Skip words that have already been processed and saved in the cache
    if word in predictions:        
        continue

    start = time.time()
    rows = []
    for inst in sample_instances(word):
        ctx = context_window(inst["tagged"])
        answers = {method: predict(method, ctx, word) for method in ALL_METHODS}
        rows.append({"cui": inst["cui"], "answers": answers})

    predictions[word] = rows
    CACHE.write_text(json.dumps(predictions))  # save after every word
    print(f"{word:16} {len(rows):3} abstracts  {time.time() - start:6.1f} s")

print("Done.")

Ca                40 abstracts   251.6 s
DDD               20 abstracts     1.0 s
DON               20 abstracts   361.4 s
FAS               20 abstracts    57.6 s
GAG               20 abstracts   235.8 s
MRS               20 abstracts    61.1 s
PVC               20 abstracts   219.1 s
RA                30 abstracts   300.9 s
SARS              20 abstracts   106.4 s
Staph             20 abstracts    90.0 s
TNT               20 abstracts    63.3 s
US                20 abstracts    64.7 s
B-Cell Leukemia   20 abstracts     1.3 s
Borrelia          20 abstracts    96.4 s
Callus            20 abstracts   234.9 s
Cell              20 abstracts   536.8 s
Cement            20 abstracts   409.9 s
Cilia             20 abstracts    83.8 s
ERUPTION          20 abstracts   246.8 s
Exercises         20 abstracts   482.5 s
Follicles         20 abstracts    56.4 s
Ganglion          20 abstracts    42.8 s
Medullary         20 abstracts     8.6 s
Milk              20 abstracts   408.8 s
Nurse           